# Análise Exploratória das Features Numéricas

Este notebook gera o HTML `features_analise_exploratoria.html` e também exporta artefatos prontos para LaTeX na pasta `features_analise_exploratoria/`: tabelas `.tex`, figuras `.pdf` e `.png`, além de um arquivo com comandos de exemplo para inserir no TCC.

In [1]:

# ============================================================
# ANÁLISE EXPLORATÓRIA DAS FEATURES NUMÉRICAS
# Versão v15 - HTML + EXPORTAÇÃO PARA LaTeX
#
# Saídas principais:
# - features_analise_exploratoria_v15.html
#
# Pasta para LaTeX:
# - features_analise_exploratoria/
#   ├── tabela_dados_gerais.tex
#   ├── tabela_normalidade.tex
#   ├── tabela_resumo_normalidade.tex
#   ├── grafico_curtose.pdf
#   ├── grafico_curtose.png
#   ├── matriz_pearson.pdf
#   ├── matriz_pearson.png
#   ├── matriz_spearman.pdf
#   ├── matriz_spearman.png
#   └── comandos_latex_exemplo.tex
#
# Ajustes principais:
# - Continua gerando HTML para análise visual
# - Exporta tabelas em .tex para usar direto no LaTeX
# - Exporta gráficos/matrizes em .pdf e .png
# - Não depende de clicar em botões no HTML para levar ao TCC
# ============================================================

import io
import base64
import warnings
import html
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")


# ============================================================
# CONFIGURAÇÕES
# ============================================================

ARQUIVO_DADOS = "creditcard.csv"
PASTA_SAIDA = "."
ARQUIVO_HTML_SAIDA = "features_analise_exploratoria.html"
PASTA_LATEX = "features_analise_exploratoria"


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def formatar_int(valor):
    try:
        return f"{int(valor):,}".replace(",", ".")
    except Exception:
        return str(valor)


def formatar_float(valor, casas=6):
    if pd.isna(valor):
        return "-"

    try:
        return f"{float(valor):.{casas}f}".replace(".", ",")
    except Exception:
        return str(valor)


def formatar_float_latex(valor, casas=6):
    if pd.isna(valor):
        return "-"

    try:
        return f"{float(valor):.{casas}f}"
    except Exception:
        return str(valor)


def formatar_pvalor(valor):
    """
    Formata p-valor em notação científica.

    Em bases grandes, o p-valor pode ser tão pequeno que o SciPy retorna 0.0
    por limite de precisão numérica. Nesse caso, mostramos como menor que 1e-300.
    """

    if pd.isna(valor):
        return "-"

    try:
        valor = float(valor)
    except Exception:
        return str(valor)

    if valor == 0:
        return "< 1e-300"

    return f"{valor:.6e}"


def formatar_pvalor_latex(valor):
    """
    Versão segura para LaTeX.
    """

    if pd.isna(valor):
        return "-"

    try:
        valor = float(valor)
    except Exception:
        return str(valor)

    if valor == 0:
        return r"$< 10^{-300}$"

    mantissa, expoente = f"{valor:.3e}".split("e")
    expoente = int(expoente)

    return rf"${mantissa} \times 10^{{{expoente}}}$"


def detectar_target(df):
    if "status_fraude" in df.columns:
        return "status_fraude"

    if "Class" in df.columns:
        return "Class"

    return None


def decisao_normalidade(pvalor, alpha):
    if pd.isna(pvalor):
        return "Indefinido"

    if pvalor < alpha:
        return "Rejeita normalidade"

    return "Não rejeita normalidade"


def teste_normalidade_ks_base_inteira(x):
    """
    Teste de normalidade Kolmogorov-Smirnov após padronização interna.

    H0: a variável segue distribuição normal.
    H1: a variável não segue distribuição normal.
    """

    x = pd.Series(x).dropna().astype(float)

    if len(x) < 3:
        return np.nan, np.nan

    desvio = x.std(ddof=1)

    if desvio == 0 or pd.isna(desvio):
        return np.nan, np.nan

    z = (x - x.mean()) / desvio

    estatistica, pvalor = stats.kstest(
        z,
        "norm"
    )

    return float(estatistica), float(pvalor)


def classificar_curtose(curtose):
    """
    Classificação usando curtose excessiva, padrão retornado por pandas.kurtosis().
    """

    if pd.isna(curtose):
        return "Indefinida"

    if curtose > 1:
        return "Leptocúrtica - caudas pesadas"

    if curtose < -1:
        return "Platicúrtica - caudas leves"

    return "Aproximadamente mesocúrtica"


def fig_to_base64(fig):
    buffer = io.BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=150,
        bbox_inches="tight"
    )

    buffer.seek(0)

    return base64.b64encode(buffer.read()).decode("utf-8")


def gerar_grafico_pizza_curtose_fig(tabela_dados_gerais):
    """
    Retorna figura matplotlib do gráfico de pizza com a distribuição
    percentual dos tipos de curtose.
    """

    if (
        tabela_dados_gerais is None
        or tabela_dados_gerais.empty
        or "Classificacao_Curtose" not in tabela_dados_gerais.columns
    ):
        return None

    contagem = (
        tabela_dados_gerais["Classificacao_Curtose"]
        .value_counts()
        .sort_values(ascending=False)
    )

    if contagem.empty:
        return None

    labels = contagem.index.tolist()
    values = contagem.values

    mapa_cores = {
        "Leptocúrtica - caudas pesadas": "#2563eb",
        "Aproximadamente mesocúrtica": "#f97316",
        "Platicúrtica - caudas leves": "#16a34a",
        "Indefinida": "#64748b"
    }

    cores = [
        mapa_cores.get(label, "#64748b")
        for label in labels
    ]

    fig, ax = plt.subplots(figsize=(10, 8))

    wedges, texts, autotexts = ax.pie(
        values,
        labels=labels,
        colors=cores,
        autopct="%1.1f%%",
        startangle=90,
        pctdistance=0.62,
        labeldistance=1.10,
        textprops={
            "fontsize": 12,
            "fontweight": "bold"
        },
        wedgeprops={
            "linewidth": 1.2,
            "edgecolor": "white"
        }
    )

    for text, cor in zip(texts, cores):
        text.set_color(cor)
        text.set_fontweight("bold")
        text.set_fontsize(13)

    for autotext, cor in zip(autotexts, cores):
        autotext.set_color(cor)
        autotext.set_fontweight("bold")
        autotext.set_fontsize(12)
        autotext.set_bbox(
            dict(
                boxstyle="round,pad=0.22",
                facecolor="white",
                edgecolor=cor,
                linewidth=1.1,
                alpha=0.92
            )
        )

    ax.set_title(
        "Distribuição dos Tipos de Curtose",
        fontsize=16,
        fontweight="bold",
        pad=18
    )

    ax.axis("equal")

    plt.tight_layout()

    return fig


def gerar_tabela_html(df, table_id, classe="data-table", max_linhas=None):
    if df is None or df.empty:
        return "<p>Nenhum dado disponível.</p>"

    df_html = df.copy()

    if max_linhas is not None:
        df_html = df_html.head(max_linhas)

    html_tabela = f'<table id="{table_id}" class="{classe}">\n'
    html_tabela += "<thead><tr>"

    for col in df_html.columns:
        html_tabela += f"<th>{html.escape(str(col))}</th>"

    html_tabela += "</tr></thead>\n<tbody>\n"

    for _, row in df_html.iterrows():
        html_tabela += "<tr>"

        for valor in row:
            html_tabela += f"<td>{html.escape(str(valor))}</td>"

        html_tabela += "</tr>\n"

    html_tabela += "</tbody></table>"

    return html_tabela


def cor_correlacao(valor):
    """
    Paleta:
    vermelho = negativo forte
    branco = próximo de zero
    azul = positivo forte
    """

    if pd.isna(valor):
        return "#f8fafc"

    valor = float(valor)

    if valor >= 0.90:
        return "#08306b"
    elif valor >= 0.70:
        return "#08519c"
    elif valor >= 0.50:
        return "#2171b5"
    elif valor >= 0.30:
        return "#6baed6"
    elif valor >= 0.10:
        return "#c6dbef"
    elif valor > -0.10:
        return "#f8fafc"
    elif valor > -0.30:
        return "#fee2e2"
    elif valor > -0.50:
        return "#fecaca"
    elif valor > -0.70:
        return "#f87171"
    elif valor > -0.90:
        return "#dc2626"
    else:
        return "#7f1d1d"


def gerar_escala_correlacao_html():
    return """
    <div class="scale-box">
        <div class="scale-title">Escala de cor da correlação</div>
        <div class="scale-bar"></div>
        <div class="scale-labels">
            <span>-1</span>
            <span>0</span>
            <span>+1</span>
        </div>
    </div>
    """


def gerar_matriz_html(matriz, titulo, table_id):
    if matriz is None or matriz.empty:
        return f"""
        <section class="section">
            <h2>{html.escape(titulo)}</h2>
            <p>Nenhuma matriz disponível.</p>
        </section>
        """

    cols = matriz.columns.tolist()

    tabela = f'<div class="matrix-scroll"><table id="{table_id}" class="corr-table">\n'
    tabela += "<thead><tr><th>Variável</th>"

    for col in cols:
        tabela += f"<th>{html.escape(str(col))}</th>"

    tabela += "</tr></thead><tbody>"

    for idx, row in matriz.iterrows():
        tabela += f"<tr><th>{html.escape(str(idx))}</th>"

        for col in cols:
            val = row[col]
            bg = cor_correlacao(val)
            texto = "" if pd.isna(val) else f"{float(val):.3f}"

            if pd.isna(val):
                color = "#020617"
            else:
                color = "#ffffff" if abs(float(val)) >= 0.70 else "#020617"

            tabela += (
                f'<td style="background:{bg}; color:{color};">'
                f'{texto}</td>'
            )

        tabela += "</tr>"

    tabela += "</tbody></table></div>"

    escala = gerar_escala_correlacao_html()

    return f"""
    <section class="section">
        <div class="section-header">
            <h2>{html.escape(titulo)}</h2>
            <button class="download-btn" onclick="baixarTabelaCSV('{table_id}', '{table_id}.csv')">
                Baixar CSV
            </button>
        </div>
        {escala}
        {tabela}
    </section>
    """


def gerar_card(titulo, valor, subtitulo=None):
    subtitulo_html = ""

    if subtitulo is not None:
        subtitulo_html = f'<div class="card-subtitle">{subtitulo}</div>'

    return f"""
    <div class="metric-card">
        <div class="card-title">{titulo}</div>
        <div class="card-value">{valor}</div>
        {subtitulo_html}
    </div>
    """


def preparar_df_formatado_html(df):
    """
    Formata tabelas para exibição no HTML.
    """

    df_fmt = df.copy()

    for col in df_fmt.columns:
        if col == "P_Valor":
            df_fmt[col] = df_fmt[col].apply(formatar_pvalor)

        elif pd.api.types.is_float_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].apply(lambda x: formatar_float(x, 6))

        elif pd.api.types.is_integer_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].apply(formatar_int)

    return df_fmt


def preparar_df_formatado_latex(df):
    """
    Formata tabelas para exportação LaTeX.
    Mantém ponto decimal, pois é mais seguro para LaTeX e reprodutibilidade.
    """

    df_fmt = df.copy()

    for col in df_fmt.columns:
        if col == "P_Valor":
            df_fmt[col] = df_fmt[col].apply(formatar_pvalor_latex)

        elif pd.api.types.is_float_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].apply(lambda x: formatar_float_latex(x, 6))

        elif pd.api.types.is_integer_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].apply(lambda x: int(x) if not pd.isna(x) else x)

    return df_fmt


def salvar_tabela_latex(df, caminho, caption, label, longtable=True):
    """
    Salva tabela em LaTeX.

    Requer no preâmbulo do LaTeX:
    \\usepackage{booktabs}
    \\usepackage{longtable}
    \\usepackage{float}
    \\usepackage{graphicx}
    """

    caminho = Path(caminho)

    tex = df.to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label
    )

    caminho.write_text(tex, encoding="utf-8")


def gerar_figura_matriz_correlacao(matriz, titulo):
    """
    Gera imagem da matriz de correlação para LaTeX.
    """

    fig, ax = plt.subplots(figsize=(13, 11))

    im = ax.imshow(
        matriz.values,
        vmin=-1,
        vmax=1,
        cmap="coolwarm"
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )

    cbar.set_label(
        "Correlação",
        fontsize=12,
        fontweight="bold"
    )

    labels = matriz.columns.tolist()

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))

    ax.set_xticklabels(
        labels,
        rotation=90,
        fontsize=8
    )

    ax.set_yticklabels(
        labels,
        fontsize=8
    )

    ax.set_title(
        titulo,
        fontsize=15,
        fontweight="bold",
        pad=16
    )

    plt.tight_layout()

    return fig


def exportar_artefatos_latex(
    pasta_latex,
    tabela_dados_gerais,
    tabela_normalidade,
    resumo_normalidade,
    matriz_pearson,
    matriz_spearman,
    grafico_curtose_fig
):
    """
    Exporta tabelas e figuras prontas para LaTeX.
    """

    pasta_latex = Path(pasta_latex)
    pasta_latex.mkdir(parents=True, exist_ok=True)

    tabela_dados_gerais_latex = preparar_df_formatado_latex(tabela_dados_gerais)
    tabela_normalidade_latex = preparar_df_formatado_latex(tabela_normalidade)
    resumo_normalidade_latex = preparar_df_formatado_latex(resumo_normalidade)

    salvar_tabela_latex(
        tabela_dados_gerais_latex,
        pasta_latex / "tabela_dados_gerais.tex",
        caption="Estatísticas descritivas das features numéricas.",
        label="tab:dados-gerais-features",
        longtable=True
    )

    salvar_tabela_latex(
        tabela_normalidade_latex,
        pasta_latex / "tabela_normalidade.tex",
        caption="Teste de normalidade Kolmogorov-Smirnov aplicado às features numéricas.",
        label="tab:normalidade-features",
        longtable=True
    )

    salvar_tabela_latex(
        resumo_normalidade_latex,
        pasta_latex / "tabela_resumo_normalidade.tex",
        caption="Resumo das decisões do teste de normalidade por nível de significância.",
        label="tab:resumo-normalidade",
        longtable=False
    )

    if grafico_curtose_fig is not None:
        grafico_curtose_fig.savefig(
            pasta_latex / "grafico_curtose.pdf",
            bbox_inches="tight"
        )

        grafico_curtose_fig.savefig(
            pasta_latex / "grafico_curtose.png",
            dpi=300,
            bbox_inches="tight"
        )

    fig_pearson = gerar_figura_matriz_correlacao(
        matriz_pearson,
        "Matriz de Correlação de Pearson"
    )

    fig_pearson.savefig(
        pasta_latex / "matriz_pearson.pdf",
        bbox_inches="tight"
    )

    fig_pearson.savefig(
        pasta_latex / "matriz_pearson.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig_pearson)

    fig_spearman = gerar_figura_matriz_correlacao(
        matriz_spearman,
        "Matriz de Correlação de Spearman"
    )

    fig_spearman.savefig(
        pasta_latex / "matriz_spearman.pdf",
        bbox_inches="tight"
    )

    fig_spearman.savefig(
        pasta_latex / "matriz_spearman.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig_spearman)

    comandos_latex = r"""
% ============================================================
% COMANDOS LaTeX DE EXEMPLO
% Recomendado no preâmbulo:
%
% \usepackage{graphicx}
% \usepackage{float}
% \usepackage{booktabs}
% \usepackage{longtable}
% \usepackage{pdflscape}
% ============================================================

% ----------------------------
% Tabela resumo de normalidade
% ----------------------------
\begin{table}[H]
\centering
\caption{Resumo das decisões do teste de normalidade.}
\label{tab:resumo-normalidade-main}
\input{features_analise_exploratoria/tabela_resumo_normalidade.tex}
\end{table}

% ----------------------------
% Tabela de dados gerais
% ----------------------------
% Como é uma tabela larga, recomenda-se usar landscape:
\begin{landscape}
\small
\input{features_analise_exploratoria/tabela_dados_gerais.tex}
\end{landscape}

% ----------------------------
% Tabela de normalidade
% ----------------------------
\begin{landscape}
\small
\input{features_analise_exploratoria/tabela_normalidade.tex}
\end{landscape}

% ----------------------------
% Gráfico de curtose
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=0.75\textwidth]{features_analise_exploratoria/grafico_curtose.pdf}
\caption{Distribuição dos tipos de curtose das features numéricas.}
\label{fig:grafico-curtose}
\end{figure}

% ----------------------------
% Matriz de Pearson
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{features_analise_exploratoria/matriz_pearson.pdf}
\caption{Matriz de correlação de Pearson entre as features numéricas.}
\label{fig:matriz-pearson}
\end{figure}

% ----------------------------
% Matriz de Spearman
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{features_analise_exploratoria/matriz_spearman.pdf}
\caption{Matriz de correlação de Spearman entre as features numéricas.}
\label{fig:matriz-spearman}
\end{figure}
"""

    (pasta_latex / "comandos_latex_exemplo.tex").write_text(
        comandos_latex,
        encoding="utf-8"
    )


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def gerar_analise_exploratoria_features(
    arquivo_dados=ARQUIVO_DADOS,
    pasta_saida=PASTA_SAIDA,
    arquivo_html_saida=ARQUIVO_HTML_SAIDA,
    pasta_latex=PASTA_LATEX,
    exportar_latex=True
):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_html = pasta_saida / arquivo_html_saida
    caminho_latex = pasta_saida / pasta_latex

    df = pd.read_csv(arquivo_dados)

    target_name = detectar_target(df)

    features_numericas = [
        col for col in df.columns
        if col != target_name
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    if len(features_numericas) == 0:
        raise ValueError("Nenhuma feature numérica encontrada após remover o rótulo.")

    linhas_dados_gerais = []
    linhas_normalidade = []

    for feature in features_numericas:
        x = df[feature].dropna().astype(float)

        minimo = x.min()
        maximo = x.max()
        amplitude = maximo - minimo
        media = x.mean()
        mediana = x.median()
        desvio_padrao = x.std(ddof=1)

        p25 = x.quantile(0.25)
        p27 = x.quantile(0.27)
        p75 = x.quantile(0.75)

        assimetria = x.skew()
        curtose = x.kurtosis()
        classificacao_curtose = classificar_curtose(curtose)

        ks_estatistica, pvalor = teste_normalidade_ks_base_inteira(x)

        linhas_dados_gerais.append({
            "Variavel": feature,
            "Minimo": minimo,
            "Maximo": maximo,
            "Amplitude": amplitude,
            "Media": media,
            "Mediana": mediana,
            "Desvio_Padrao": desvio_padrao,
            "P25": p25,
            "P27": p27,
            "P75": p75,
            "Assimetria": assimetria,
            "Curtose": curtose,
            "Classificacao_Curtose": classificacao_curtose,
        })

        linhas_normalidade.append({
            "Variavel": feature,
            "Teste_Normalidade": "Kolmogorov-Smirnov",
            "KS_Estatistica": ks_estatistica,
            "P_Valor": pvalor,
            "Decisao_Normalidade_Alpha_0_001": decisao_normalidade(pvalor, 0.001),
            "Decisao_Normalidade_Alpha_0_01": decisao_normalidade(pvalor, 0.01),
            "Decisao_Normalidade_Alpha_0_05": decisao_normalidade(pvalor, 0.05),
            "Decisao_Normalidade_Alpha_0_10": decisao_normalidade(pvalor, 0.10),
        })

    tabela_dados_gerais = pd.DataFrame(linhas_dados_gerais)
    tabela_normalidade = pd.DataFrame(linhas_normalidade)

    colunas_dados_gerais = [
        "Variavel",
        "Minimo",
        "Maximo",
        "Amplitude",
        "Media",
        "Mediana",
        "Desvio_Padrao",
        "P25",
        "P27",
        "P75",
        "Assimetria",
        "Curtose",
        "Classificacao_Curtose",
    ]

    colunas_normalidade = [
        "Variavel",
        "Teste_Normalidade",
        "KS_Estatistica",
        "P_Valor",
        "Decisao_Normalidade_Alpha_0_001",
        "Decisao_Normalidade_Alpha_0_01",
        "Decisao_Normalidade_Alpha_0_05",
        "Decisao_Normalidade_Alpha_0_10",
    ]

    tabela_dados_gerais = tabela_dados_gerais[colunas_dados_gerais]
    tabela_normalidade = tabela_normalidade[colunas_normalidade]

    # ========================================================
    # MATRIZES
    # ========================================================

    df_features = df[features_numericas].copy()

    matriz_pearson = df_features.corr(method="pearson")
    matriz_spearman = df_features.corr(method="spearman")

    # ========================================================
    # RESUMO GERAL
    # ========================================================

    total_features = len(features_numericas)

    rejeitam_0001 = int((tabela_normalidade["P_Valor"] < 0.001).sum())
    rejeitam_001 = int((tabela_normalidade["P_Valor"] < 0.01).sum())
    rejeitam_005 = int((tabela_normalidade["P_Valor"] < 0.05).sum())
    rejeitam_010 = int((tabela_normalidade["P_Valor"] < 0.10).sum())

    nao_rejeitam_0001 = int((tabela_normalidade["P_Valor"] >= 0.001).sum())
    nao_rejeitam_001 = int((tabela_normalidade["P_Valor"] >= 0.01).sum())
    nao_rejeitam_005 = int((tabela_normalidade["P_Valor"] >= 0.05).sum())
    nao_rejeitam_010 = int((tabela_normalidade["P_Valor"] >= 0.10).sum())

    resumo_normalidade = pd.DataFrame([
        {
            "Alpha": "0.001",
            "Rejeitam_Normalidade": rejeitam_0001,
            "Nao_Rejeitam_Normalidade": nao_rejeitam_0001
        },
        {
            "Alpha": "0.01",
            "Rejeitam_Normalidade": rejeitam_001,
            "Nao_Rejeitam_Normalidade": nao_rejeitam_001
        },
        {
            "Alpha": "0.05",
            "Rejeitam_Normalidade": rejeitam_005,
            "Nao_Rejeitam_Normalidade": nao_rejeitam_005
        },
        {
            "Alpha": "0.10",
            "Rejeitam_Normalidade": rejeitam_010,
            "Nao_Rejeitam_Normalidade": nao_rejeitam_010
        }
    ])

    # ========================================================
    # FIGURAS
    # ========================================================

    grafico_curtose_fig = gerar_grafico_pizza_curtose_fig(tabela_dados_gerais)

    if grafico_curtose_fig is not None:
        grafico_pizza_curtose = fig_to_base64(grafico_curtose_fig)
    else:
        grafico_pizza_curtose = None

    # ========================================================
    # EXPORTAÇÃO LaTeX
    # ========================================================

    if exportar_latex:
        exportar_artefatos_latex(
            pasta_latex=caminho_latex,
            tabela_dados_gerais=tabela_dados_gerais,
            tabela_normalidade=tabela_normalidade,
            resumo_normalidade=resumo_normalidade,
            matriz_pearson=matriz_pearson,
            matriz_spearman=matriz_spearman,
            grafico_curtose_fig=grafico_curtose_fig
        )

    if grafico_curtose_fig is not None:
        plt.close(grafico_curtose_fig)

    # ========================================================
    # HTML
    # ========================================================

    tabela_dados_gerais_fmt = preparar_df_formatado_html(tabela_dados_gerais)
    tabela_normalidade_fmt = preparar_df_formatado_html(tabela_normalidade)

    tabela_dados_gerais_html = gerar_tabela_html(
        tabela_dados_gerais_fmt,
        table_id="tabela_dados_gerais"
    )

    tabela_normalidade_html = gerar_tabela_html(
        tabela_normalidade_fmt,
        table_id="tabela_normalidade"
    )

    matriz_pearson_html = gerar_matriz_html(
        matriz_pearson,
        "Matriz de Correlação de Pearson",
        table_id="matriz_pearson"
    )

    matriz_spearman_html = gerar_matriz_html(
        matriz_spearman,
        "Matriz de Correlação de Spearman",
        table_id="matriz_spearman"
    )

    if grafico_pizza_curtose is not None:
        grafico_pizza_curtose_html = f"""
        <section class="section">
            <div class="section-actions-only">
                <a
                    class="download-btn link-btn"
                    href="data:image/png;base64,{grafico_pizza_curtose}"
                    download="distribuicao_tipos_curtose.png"
                >
                    Baixar PNG
                </a>
            </div>
            <img
                id="grafico_curtose"
                class="plot-img"
                src="data:image/png;base64,{grafico_pizza_curtose}"
                alt="Distribuição dos Tipos de Curtose"
            >
        </section>
        """
    else:
        grafico_pizza_curtose_html = ""

    pasta_latex_html = html.escape(str(caminho_latex))

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Análise Exploratória das Features</title>

        <style>
            body {{
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
                margin: 0;
                padding: 32px;
            }}

            .container {{
                max-width: 1450px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                margin-bottom: 34px;
                color: #020617;
                font-size: 34px;
            }}

            .section {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 28px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .section h2 {{
                text-align: center;
                margin-top: 0;
                margin-bottom: 22px;
                font-size: 24px;
                color: #020617;
            }}

            .section-header {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 18px;
            }}

            .section-header h2 {{
                margin: 0;
            }}

            .section-actions-only {{
                display: flex;
                justify-content: flex-end;
                margin-bottom: 12px;
            }}

            .download-btn {{
                border: 1px solid #bfdbfe;
                background: #eff6ff;
                color: #1e3a8a;
                padding: 8px 12px;
                border-radius: 10px;
                font-weight: 800;
                font-size: 13px;
                cursor: pointer;
                text-decoration: none;
                display: inline-block;
            }}

            .download-btn:hover {{
                background: #dbeafe;
            }}

            .metric-grid {{
                display: grid;
                grid-template-columns: repeat(5, 1fr);
                gap: 14px;
                margin-top: 18px;
            }}

            .metric-card {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
            }}

            .card-title {{
                font-size: 13px;
                font-weight: 800;
                color: #475569;
                margin-bottom: 8px;
            }}

            .card-value {{
                font-size: 22px;
                font-weight: 900;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
            }}

            .card-subtitle {{
                font-size: 12px;
                color: #64748b;
                margin-top: 8px;
                line-height: 1.35;
            }}

            .table-wrapper {{
                overflow-x: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 620px;
                overflow-y: auto;
            }}

            .data-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 13px;
                margin-top: 0;
            }}

            .data-table th {{
                background: #0f172a;
                color: white;
                padding: 10px 8px;
                text-align: left;
                position: sticky;
                top: 0;
                z-index: 1;
            }}

            .data-table td {{
                border-bottom: 1px solid #e2e8f0;
                padding: 8px;
                color: #020617;
                white-space: nowrap;
            }}

            .data-table tr:nth-child(even) {{
                background: #f8fafc;
            }}

            .matrix-scroll {{
                overflow: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 720px;
            }}

            .corr-table {{
                border-collapse: collapse;
                font-size: 11px;
                min-width: 100%;
            }}

            .corr-table th {{
                background: #0f172a;
                color: white;
                padding: 7px;
                position: sticky;
                top: 0;
                z-index: 2;
            }}

            .corr-table th:first-child {{
                left: 0;
                z-index: 3;
            }}

            .corr-table td {{
                text-align: center;
                padding: 6px;
                border: 1px solid #e2e8f0;
                font-weight: 700;
                min-width: 58px;
            }}

            .scale-box {{
                max-width: 560px;
                margin: 0 auto 18px auto;
            }}

            .scale-title {{
                text-align: center;
                font-size: 13px;
                font-weight: 800;
                color: #334155;
                margin-bottom: 6px;
            }}

            .scale-bar {{
                height: 18px;
                border-radius: 999px;
                border: 1px solid #cbd5e1;
                background: linear-gradient(
                    to right,
                    #7f1d1d 0%,
                    #dc2626 15%,
                    #f87171 30%,
                    #fee2e2 42%,
                    #f8fafc 50%,
                    #c6dbef 58%,
                    #6baed6 70%,
                    #2171b5 82%,
                    #08519c 92%,
                    #08306b 100%
                );
            }}

            .scale-labels {{
                display: flex;
                justify-content: space-between;
                font-size: 12px;
                font-weight: 700;
                color: #475569;
                margin-top: 5px;
            }}

            .plot-img {{
                display: block;
                max-width: 100%;
                margin: 0 auto;
                border-radius: 12px;
                border: 1px solid #e2e8f0;
            }}

            .latex-box {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
                color: #334155;
                line-height: 1.55;
                font-size: 14px;
            }}

            .code {{
                font-family: Consolas, Monaco, monospace;
                background: #0f172a;
                color: #e2e8f0;
                padding: 10px;
                border-radius: 10px;
                overflow-x: auto;
                margin-top: 10px;
            }}

            @media (max-width: 1200px) {{
                .metric-grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}
            }}

            @media (max-width: 640px) {{
                .metric-grid {{
                    grid-template-columns: 1fr;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1>Análise Exploratória das Features Numéricas</h1>

            <section class="section">
                <h2>Resumo Geral</h2>

                <div class="metric-grid">
                    {gerar_card("Features numéricas", formatar_int(total_features), "Variável de rótulo removida da análise.")}
                    {gerar_card("Teste de normalidade", "KS", "H0: a variável segue distribuição normal.")}
                    {gerar_card("Rejeitam α = 0.001", formatar_int(rejeitam_0001), "p-valor < 0.001.")}
                    {gerar_card("Rejeitam α = 0.01", formatar_int(rejeitam_001), "p-valor < 0.01.")}
                    {gerar_card("Rejeitam α = 0.05", formatar_int(rejeitam_005), "p-valor < 0.05.")}
                    {gerar_card("Rejeitam α = 0.10", formatar_int(rejeitam_010), "p-valor < 0.10.")}
                    {gerar_card("Não rejeitam α = 0.001", formatar_int(nao_rejeitam_0001), "p-valor >= 0.001.")}
                    {gerar_card("Não rejeitam α = 0.01", formatar_int(nao_rejeitam_001), "p-valor >= 0.01.")}
                    {gerar_card("Não rejeitam α = 0.05", formatar_int(nao_rejeitam_005), "p-valor >= 0.05.")}
                    {gerar_card("Não rejeitam α = 0.10", formatar_int(nao_rejeitam_010), "p-valor >= 0.10.")}
                </div>
            </section>

            {grafico_pizza_curtose_html}

            <section class="section">
                <div class="section-header">
                    <h2>Tabela de Dados Gerais das Features</h2>
                    <button class="download-btn" onclick="baixarTabelaCSV('tabela_dados_gerais', 'tabela_dados_gerais.csv')">
                        Baixar CSV
                    </button>
                </div>
                <div class="table-wrapper">
                    {tabela_dados_gerais_html}
                </div>
            </section>

            <section class="section">
                <div class="section-header">
                    <h2>Tabela de Normalidade das Features</h2>
                    <button class="download-btn" onclick="baixarTabelaCSV('tabela_normalidade', 'tabela_normalidade.csv')">
                        Baixar CSV
                    </button>
                </div>
                <div class="table-wrapper">
                    {tabela_normalidade_html}
                </div>
            </section>

            {matriz_pearson_html}

            {matriz_spearman_html}

        </div>

        <script>
            function limparTextoCSV(texto) {{
                if (texto === null || texto === undefined) {{
                    return "";
                }}

                texto = String(texto).replace(/\\n/g, " ").replace(/\\s+/g, " ").trim();

                if (texto.includes(";") || texto.includes('"')) {{
                    texto = '"' + texto.replace(/"/g, '""') + '"';
                }}

                return texto;
            }}

            function baixarTabelaCSV(tableId, filename) {{
                const tabela = document.getElementById(tableId);

                if (!tabela) {{
                    alert("Tabela não encontrada: " + tableId);
                    return;
                }}

                const linhas = [];

                tabela.querySelectorAll("tr").forEach(function(row) {{
                    const celulas = Array.from(row.querySelectorAll("th, td"));
                    const linha = celulas.map(celula => limparTextoCSV(celula.innerText)).join(";");
                    linhas.push(linha);
                }});

                const csv = "\\ufeff" + linhas.join("\\n");
                const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
                const url = URL.createObjectURL(blob);

                const link = document.createElement("a");
                link.href = url;
                link.download = filename;
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);

                URL.revokeObjectURL(url);
            }}
        </script>
    </body>
    </html>
    """

    caminho_html.write_text(
        html_final,
        encoding="utf-8"
    )

    print("=" * 80)
    print("ANÁLISE EXPLORATÓRIA FINALIZADA")
    print("=" * 80)
    print(f"HTML salvo em: {caminho_html.resolve()}")

    if exportar_latex:
        print(f"Arquivos LaTeX salvos em: {caminho_latex.resolve()}")

    print(f"Features numéricas analisadas: {total_features}")
    print("=" * 80)

    return {
        "tabela_dados_gerais": tabela_dados_gerais,
        "tabela_normalidade": tabela_normalidade,
        "resumo_normalidade": resumo_normalidade,
        "matriz_pearson": matriz_pearson,
        "matriz_spearman": matriz_spearman,
        "caminho_html": caminho_html,
        "caminho_latex": caminho_latex
    }


# ============================================================
# EXECUÇÃO
# ============================================================

resultado_eda_features = gerar_analise_exploratoria_features(
    arquivo_dados="creditcard.csv",
    pasta_saida=".",
    arquivo_html_saida="features_analise_exploratoria.html",
    pasta_latex="features_analise_exploratoria",
    exportar_latex=True
)

resultado_eda_features["tabela_dados_gerais"].head()


ANÁLISE EXPLORATÓRIA FINALIZADA
HTML salvo em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\features_analise_exploratoria.html
Arquivos LaTeX salvos em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\features_analise_exploratoria
Features numéricas analisadas: 30


,Variavel,Minimo,Maximo,Amplitude,Media,Mediana,Desvio_Padrao,P25,P27,P75,Assimetria,Curtose,Classificacao_Curtose
0,tempo_desde_a_primeira_transacao,0.000000,172792.000000,172792.000000,94811.077600,84692.500000,47481.047891,54204.750000,56800.000000,139298.000000,-0.035581,-1.293432,Platicúrtica - caudas leves
1,V1,-56.407510,2.454930,58.862440,0.005917,0.020384,1.948026,-0.915951,-0.843297,1.316068,-3.273271,32.727332,Leptocúrtica - caudas pesadas
2,V2,-72.715728,22.057729,94.773457,-0.004135,0.063949,1.646703,-0.600321,-0.533520,0.800283,-4.695162,96.898173,Leptocúrtica - caudas pesadas
3,V3,-48.325589,9.382558,57.708148,0.001613,0.179963,1.508682,-0.889682,-0.772765,1.026960,-2.151984,25.186530,Leptocúrtica - caudas pesadas
4,V4,-5.683171,16.875344,22.558515,-0.002966,-0.022248,1.414184,-0.850134,-0.783611,0.739647,0.671504,2.618780,Leptocúrtica - caudas pesadas
